In [1]:
import os
import time
import threading
import psutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import intel_extension_for_pytorch as ipex
from torch.profiler import profile, record_function, ProfilerActivity

def measure_memory(pid, stop_event, memory_records):
    process = psutil.Process(pid)
    while not stop_event.is_set():
        memory_records.append(process.memory_info().rss)
        time.sleep(0.01)

def run_with_memory_tracking(func, *args, **kwargs):
    memory_records = []
    stop_event = threading.Event()
    pid = os.getpid()
    tracker_thread = threading.Thread(target=measure_memory, args=(pid, stop_event, memory_records))
    
    tracker_thread.start()
    start_time = time.time()
    result = func(*args, **kwargs)
    end_time = time.time()
    stop_event.set()
    tracker_thread.join()
    
    peak_rss = max(memory_records) if memory_records else psutil.Process(pid).memory_info().rss
    duration = end_time - start_time
    return result, duration, peak_rss

def profile_model(model, name, inputs):
    with profile(activities=[ProfilerActivity.CPU], record_shapes=True, profile_memory=True, with_stack=True) as prof:
        with record_function("model_inference"):
            with torch.inference_mode():
                model.generate(**inputs, max_new_tokens=3, min_new_tokens=3)
    
    prof.export_chrome_trace(f"trace_{name}.json")
    print(f"--- Profile for {name} ---")
    print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))

[W522 10:30:57.923588918 OperatorEntry.cpp:218] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::_addmm_activation(Tensor self, Tensor mat1, Tensor mat2, *, Scalar beta=1, Scalar alpha=1, bool use_gelu=False) -> Tensor
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: AutocastCPU
  previous kernel: registered at /pytorch/aten/src/ATen/autocast_mode.cpp:327
       new kernel: registered at /opt/workspace/ipex-cpu-dev/csrc/cpu/autocast/autocast_mode.cpp:112 (function operator())


In [2]:
model_id = "Qwen/Qwen3-4B-Thinking-2507"
tokenizer = AutoTokenizer.from_pretrained(model_id)

input_ids = torch.randint(1, 1000, (1, 1000), dtype=torch.long)
attention_mask = torch.ones((1, 1000), dtype=torch.long)
inputs = {"input_ids": input_ids, "attention_mask": attention_mask}

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True).to("cpu")
model.eval()

2026-05-22 10:30:58,283 - _http.py - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [3]:
def generate_fp16():
    with torch.inference_mode():
        return model.generate(**inputs, max_new_tokens=3, min_new_tokens=3)

_, time_fp16, mem_fp16 = run_with_memory_tracking(generate_fp16)
print(f"FP16 - Time: {time_fp16:.4f}s, Peak RSS: {mem_fp16 / (1024**2):.2f} MB")
profile_model(model, "fp16", inputs)

FP16 - Time: 38.5342s, Peak RSS: 9403.70 MB
--- Profile for fp16 ---
--------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
--------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             model_inference         0.26%     105.361ms       100.00%       40.540s       40.540s           0 B     -11.83 GB             1  
                                aten::matmul         0.05%      19.893ms        95.29%       38.632s      39.501ms       6.92 GB           0 B           978  
                                aten::linear         0.01%       3.635ms        93.10%       37.743s      49.727ms       2.06 GB        

In [7]:
qconfig_mapping = ipex.quantization.get_weight_only_quant_qconfig_mapping()

trace_input_ids = torch.randint(1, 1000, (1, 1), dtype=torch.long)
trace_attention_mask = torch.ones((1, 1), dtype=torch.long)
trace_inputs = (trace_input_ids, trace_attention_mask)

prepared_model = ipex.quantization.prepare(model, qconfig_mapping, example_inputs=trace_inputs, inplace=True)
model_int8 = ipex.quantization.convert(prepared_model, inplace=True)

def generate_int8():
    with torch.inference_mode():
        return model_int8.generate(**inputs, max_new_tokens=3, min_new_tokens=3)

_, time_int8, mem_int8 = run_with_memory_tracking(generate_int8)
print(f"INT8 - Time: {time_int8:.4f}s, Peak RSS: {mem_int8 / (1024**2):.2f} MB")
profile_model(model_int8, "int8", inputs)

2026-05-22 10:46:19,977 - _logger.py - IPEX - WARNING - [NotSupported]BatchNorm folding failed during the prepare process.


INT8 - Time: 51.9381s, Peak RSS: 21577.54 MB
--- Profile for int8 ---
--------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
--------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             model_inference         0.29%     157.082ms       100.00%       54.170s       54.170s           0 B     -11.83 GB             1  
                 torch_ipex::ipex_woq_linear        10.45%        5.659s        94.38%       51.127s      67.361ms       2.06 GB    -157.32 GB           759  
                                aten::matmul         0.02%       9.835ms        71.77%       38.879s      39.754ms       6.92 GB       